In [302]:
import torch
import torch.nn as nn
import pickle
import numpy as np
from matplotlib import pyplot as plt
import tqdm as tqdm

In [303]:
def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

In [304]:
batch1 = unpickle(r"C:\Users\simon\Coding\ML\Prototypical NN\cifar-10-batches-py\data_batch_1")
batch2 = unpickle(r"C:\Users\simon\Coding\ML\Prototypical NN\cifar-10-batches-py\data_batch_2")
batch3 = unpickle(r"C:\Users\simon\Coding\ML\Prototypical NN\cifar-10-batches-py\data_batch_3")
batch4 = unpickle(r"C:\Users\simon\Coding\ML\Prototypical NN\cifar-10-batches-py\data_batch_4")
batch5 = unpickle(r"C:\Users\simon\Coding\ML\Prototypical NN\cifar-10-batches-py\data_batch_5")

In [305]:
unique_labels = [0,1,2,3,4,5,6,7,8,9]

all_data = np.concatenate([batch1[b"data"], batch2[b"data"], batch3[b"data"], batch4[b"data"], batch5[b"data"]], axis=0)
all_labels = batch1[b"labels"] + batch2[b"labels"] + batch3[b"labels"] + batch4[b"labels"] + batch5[b"labels"]

data = all_data.reshape(-1, 3, 32, 32)
data = torch.tensor(data, dtype=torch.float32) / 255.0
labels = torch.tensor(all_labels)

In [306]:
classes_to_index = {}
for i in unique_labels:
    classes_to_index[i] = np.where(labels == i)[0]

In [307]:
def sample_data(num_classes: int, shot_count: int):
    #hard coded classes for CIFAR10 -> 0-9
    data_ret = {}
    classes = np.random.choice(unique_labels, num_classes, replace = False)
    used_indices = []
    for i in classes:
        data_indexes = np.random.choice(classes_to_index[i], shot_count, replace = False)
        used_indices.append(data_indexes)
        image_dat = data[data_indexes]
        data_ret[i] = image_dat
    used_indices = np.concatenate(used_indices)
    return data_ret, used_indices


In [308]:
model = nn.Sequential(
nn.Conv2d(in_channels = 3, out_channels = 64, kernel_size = (3,3), padding = 1), 
nn.BatchNorm2d(64),
nn.MaxPool2d(kernel_size = (2,2)), 
nn.Conv2d(64, 64, kernel_size = (3,3), padding = 1),
nn.BatchNorm2d(64),
nn.MaxPool2d(kernel_size = (2,2)),
nn.Conv2d(64, 64, kernel_size = (3,3), padding = 1),
nn.BatchNorm2d(64),
nn.MaxPool2d(kernel_size = (2,2)),
nn.Flatten(),
nn.Linear(in_features = 1024, out_features = 64),
nn.ReLU(),
nn.Linear(64,64))



In [309]:
num_classes = 5
shot = 5
num_queries = 3
lr = 0.001
episodes = 3000
optimizer = torch.optim.Adam(model.parameters(), lr = lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=episodes/3, gamma=0.5)
for i in tqdm.tqdm(range(episodes)):
    optimizer.zero_grad()
    loss = 0
    samp = sample_data(num_classes, shot)
    support_set = samp[0]
    used_indices = samp[1]
    classes = list(support_set.keys())
    prototypes = {}
    query_set = {}
    for j in classes:
        prototype = torch.sum(model.forward(support_set[j]), dim = 0) / shot
        prototypes[j] = prototype

        valid_query_indices = np.setdiff1d(classes_to_index[j], used_indices)
        query_set[j] = data[np.random.choice(valid_query_indices, num_queries)]

    for j in classes:
        for k in query_set[j]:
            forward = model.forward(k.unsqueeze(0)).squeeze(0)  # (64,)
            dists = torch.stack([torch.dist(forward, prototypes[c]) for c in classes])
            loss += 1/(num_classes * num_queries) * (torch.dist(forward, prototypes[j]) + torch.log(torch.sum(torch.exp(-dists))))
    loss.backward()
    optimizer.step()
    scheduler.step()

100%|██████████| 3000/3000 [05:45<00:00,  8.68it/s]


In [310]:
torch.save(model, "proto.pt")